# WBCBench 2026 — Aşama 9B: ConvNeXt-Tiny Final — Seeds 1 & 7

Aşama 9A'daki kilitli protokol aynen korunur: max 12 epoch, patience 3, batch 8, LR 1e-4, AdamW wd 1e-4, validation Macro-F1 model seçimi. `phase2_test` kullanılmaz.

Kernel: **medxai_gpu**.


In [1]:
from pathlib import Path
import random,time,json,csv,gc
import numpy as np, torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets,transforms,models
from sklearn.metrics import f1_score,balanced_accuracy_score,accuracy_score

ROOT=Path.cwd()
TRAIN=ROOT/"wbc_project"/"data"/"processed"/"phase2_train"
VAL=ROOT/"wbc_project"/"data"/"processed"/"phase2_eval"
BASE=ROOT/"outputs"/"final"/"convnext_tiny"
device=torch.device("cuda")
assert TRAIN.exists() and VAL.exists() and torch.cuda.is_available()
print("GPU:",torch.cuda.get_device_name(0))


GPU: NVIDIA GeForce RTX 4060 Laptop GPU


In [2]:
train_tf=transforms.Compose([transforms.Resize((224,224)),transforms.RandomHorizontalFlip(.5),transforms.RandomRotation(10),transforms.ToTensor(),transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
val_tf=transforms.Compose([transforms.Resize((224,224)),transforms.ToTensor(),transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
train_ds=datasets.ImageFolder(TRAIN,transform=train_tf); val_ds=datasets.ImageFolder(VAL,transform=val_tf)
assert train_ds.classes==val_ds.classes and len(train_ds.classes)==13
print("Train:",len(train_ds),"Val:",len(val_ds))


Train: 49794 Val: 10700


In [3]:
MAX_EPOCHS=12; PATIENCE=3; BATCH=8; LR=1e-4; WD=1e-4
def run_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark=False; torch.backends.cudnn.deterministic=True
    g=torch.Generator(); g.manual_seed(seed)
    tr=DataLoader(train_ds,batch_size=BATCH,shuffle=True,num_workers=0,pin_memory=True,generator=g)
    va=DataLoader(val_ds,batch_size=BATCH,shuffle=False,num_workers=0,pin_memory=True)
    model=models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
    model.classifier[2]=nn.Linear(model.classifier[2].in_features,13); model=model.to(device)
    criterion=nn.CrossEntropyLoss(); opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WD)
    scaler=torch.amp.GradScaler("cuda")
    out=BASE/f"seed{seed}"; out.mkdir(parents=True,exist_ok=True)
    best_path=out/f"convnext_tiny_seed{seed}_best.pt"; best=-1.; bad=0; history=[]
    config={"model":"convnext_tiny","seed":seed,"max_epochs":12,"patience":3,"batch_size":8,"lr":LR,"weight_decay":WD,"input_size":224,"selection_metric":"validation_macro_f1","test_used":False,"classes":train_ds.classes}
    (out/"config.json").write_text(json.dumps(config,ensure_ascii=False,indent=2),encoding="utf-8")
    for epoch in range(1,MAX_EPOCHS+1):
        print("\n"+"="*65); print(f"CONVNEXT SEED {seed} — EPOCH {epoch}/12"); print("="*65)
        t=time.time(); model.train(); total=n=0
        for i,(x,y) in enumerate(tr,1):
            x=x.to(device); y=y.to(device); opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda",dtype=torch.float16): z=model(x); loss=criterion(z,y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            total+=loss.item()*y.size(0); n+=y.size(0)
            if i%1000==0: print(f" batch {i}/{len(tr)} train_loss={total/n:.4f}")
        model.eval(); ys=[]; ps=[]; vl=vn=0
        with torch.inference_mode():
            for x,y in va:
                x=x.to(device); y=y.to(device)
                with torch.autocast("cuda",dtype=torch.float16): z=model(x); loss=criterion(z,y)
                vl+=loss.item()*y.size(0); vn+=y.size(0); ys+=y.cpu().tolist(); ps+=z.argmax(1).cpu().tolist()
        f1=f1_score(ys,ps,average="macro",zero_division=0); ba=balanced_accuracy_score(ys,ps); acc=accuracy_score(ys,ps)
        row={"epoch":epoch,"train_loss":total/n,"val_loss":vl/vn,"macro_f1":f1,"balanced_acc":ba,"accuracy":acc,"minutes":(time.time()-t)/60}; history.append(row)
        print(f"Macro-F1={f1:.4f} | Balanced Acc={ba:.4f} | Acc={acc:.4f} | {row['minutes']:.1f} dk")
        if f1>best:
            best=f1; bad=0; torch.save({"model_state_dict":model.state_dict(),"classes":train_ds.classes,"seed":seed,"epoch":epoch,"val_macro_f1":best,"config":config},best_path); print("✓ Yeni best")
        else: bad+=1; print(f"İyileşme yok: {bad}/{PATIENCE}")
        (out/"history.json").write_text(json.dumps(history,indent=2),encoding="utf-8")
        with open(out/"history.csv","w",newline="",encoding="utf-8") as f:
            w=csv.DictWriter(f,fieldnames=history[0].keys()); w.writeheader(); w.writerows(history)
        if bad>=PATIENCE: print("✓ Early stopping."); break
    result={"seed":seed,"best_val_macro_f1":best,"epochs_completed":len(history),"best_checkpoint":str(best_path)}
    (out/"result.json").write_text(json.dumps(result,indent=2),encoding="utf-8")
    del model,opt,scaler,tr,va; gc.collect(); torch.cuda.empty_cache()
    return result


In [4]:
results=[]
for seed in [1,7]:
    results.append(run_seed(seed))
print("\n"+"="*70)
print("CONVNEXT-TINY FINAL — SEEDS 1 & 7 SONUCU")
print("="*70)
for r in results: print(f"Seed {r['seed']}: Best Val Macro-F1={r['best_val_macro_f1']:.4f} | Epochs={r['epochs_completed']}")
print("✓ TEST SPLIT KULLANILMADI.")



CONVNEXT SEED 1 — EPOCH 1/12
 batch 1000/6225 train_loss=0.4926
 batch 2000/6225 train_loss=0.4036
 batch 3000/6225 train_loss=0.3652
 batch 4000/6225 train_loss=0.3408
 batch 5000/6225 train_loss=0.3255
 batch 6000/6225 train_loss=0.3127
Macro-F1=0.5867 | Balanced Acc=0.5886 | Acc=0.9288 | 17.3 dk
✓ Yeni best

CONVNEXT SEED 1 — EPOCH 2/12
 batch 1000/6225 train_loss=0.2256
 batch 2000/6225 train_loss=0.2182
 batch 3000/6225 train_loss=0.2132
 batch 4000/6225 train_loss=0.2119
 batch 5000/6225 train_loss=0.2096
 batch 6000/6225 train_loss=0.2080
Macro-F1=0.6475 | Balanced Acc=0.6157 | Acc=0.9324 | 10.0 dk
✓ Yeni best

CONVNEXT SEED 1 — EPOCH 3/12
 batch 1000/6225 train_loss=0.1757
 batch 2000/6225 train_loss=0.1718
 batch 3000/6225 train_loss=0.1706
 batch 4000/6225 train_loss=0.1697
 batch 5000/6225 train_loss=0.1679
 batch 6000/6225 train_loss=0.1665
Macro-F1=0.6006 | Balanced Acc=0.5732 | Acc=0.9213 | 10.1 dk
İyileşme yok: 1/3

CONVNEXT SEED 1 — EPOCH 4/12
 batch 1000/6225 train_lo